# Autoresearch Memory Smoke Test

This notebook validates the first synthesized memory layer for `autoresearch-macos`.

It checks:

1. the `memory/` directory exists
2. `MEMORY.md` exists and links the expected topic files
3. each topic file has frontmatter with `name`, `description`, and `type`
4. operational memory files still exist:
   - `results.tsv`
   - `experiment_memory.jsonl`
5. the synthesized memory layer complements, rather than replaces, the exact run logs


In [9]:
from __future__ import annotations

from pathlib import Path
import json
import re
import sys

def is_claw_root(candidate: Path) -> bool:
    return (candidate / 'src' / 'main.py').exists() and (candidate / 'tests' / 'test_porting_workspace.py').exists()

def find_claw_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if is_claw_root(candidate):
            return candidate
        for nested in (candidate / 'claw-code', candidate / 'harness' / 'claw-code'):
            if is_claw_root(nested):
                return nested
    raise RuntimeError(f'Could not locate claw-code root from {current}')

CLAW_ROOT = find_claw_root()
WORKSPACE_ROOT = CLAW_ROOT.parent.parent
AUTORESEARCH_ROOT = WORKSPACE_ROOT / 'nodes' / 'autoresearch-macos'
MEMORY_ROOT = AUTORESEARCH_ROOT / 'memory'

print({
    'claw_root': str(CLAW_ROOT),
    'autoresearch_root': str(AUTORESEARCH_ROOT),
    'memory_root': str(MEMORY_ROOT),
})

{'claw_root': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code', 'autoresearch_root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos', 'memory_root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/memory'}


In [10]:
expected_files = [
    'MEMORY.md',
    'project_current_strategy.md',
    'project_failed_ideas.md',
    'project_promising_ideas.md',
    'reference_external_sources.md',
]
missing = [name for name in expected_files if not (MEMORY_ROOT / name).exists()]
if missing:
    raise RuntimeError(f'Missing memory files: {missing}')
sorted(path.name for path in MEMORY_ROOT.iterdir())

['MEMORY.md',
 'project_current_strategy.md',
 'project_failed_ideas.md',
 'project_promising_ideas.md',
 'reference_external_sources.md']

In [11]:
index_text = (MEMORY_ROOT / 'MEMORY.md').read_text(encoding='utf-8')
for required_link in [
    './project_current_strategy.md',
    './project_failed_ideas.md',
    './project_promising_ideas.md',
    './reference_external_sources.md',
]:
    if required_link not in index_text:
        raise RuntimeError(f'MEMORY.md is missing link: {required_link}')
print(index_text[:2000])

# Memory Index

This directory is the synthesized project memory layer for `autoresearch-macos`.

It sits between:

- raw operational records such as [`results.tsv`](../results.tsv) and [`experiment_memory.jsonl`](../experiment_memory.jsonl)
- future higher-level retrieval or wiki tooling

The goal is not to replace exact run logs. The goal is to keep a compact, maintained set of topic pages that accumulate what the manager and worker learn over time.

## Topics

- [Current strategy](./project_current_strategy.md) - Current operating model, control-plane shape, and near-term execution priorities.
- [Failed ideas](./project_failed_ideas.md) - Approaches or patterns that did not work, or that should be treated cautiously.
- [Promising ideas](./project_promising_ideas.md) - Directions worth revisiting or expanding in future experiment cycles.
- [External references](./reference_external_sources.md) - Pointers to papers, repos, dashboards, or notes that are useful but live outside the code

In [12]:
def parse_frontmatter(path: Path) -> dict[str, str]:
    text = path.read_text(encoding='utf-8')
    match = re.match(r'^---\n(.*?)\n---\n', text, re.DOTALL)
    if not match:
        raise RuntimeError(f'Missing frontmatter in {path.name}')
    result = {}
    for line in match.group(1).splitlines():
        if ':' not in line:
            continue
        key, value = line.split(':', 1)
        result[key.strip()] = value.strip()
    for field in ('name', 'description', 'type'):
        if field not in result or not result[field]:
            raise RuntimeError(f'Missing {field} in {path.name}')
    return result

metadata = {
    name: parse_frontmatter(MEMORY_ROOT / name)
    for name in expected_files
    if name != 'MEMORY.md'
}
metadata

{'project_current_strategy.md': {'name': 'Current strategy',
  'description': 'Current operating strategy for the autoresearch manager/worker system and how to use the control plane safely.',
  'type': 'project'},
 'project_failed_ideas.md': {'name': 'Failed ideas',
  'description': 'Failed or cautionary patterns for the autoresearch system, especially around control-plane validation and worker behavior.',
  'type': 'project'},
 'project_promising_ideas.md': {'name': 'Promising ideas',
  'description': 'Promising next steps for memory, orchestration, and experiment quality in autoresearch.',
  'type': 'project'},
 'reference_external_sources.md': {'name': 'External references',
  'description': 'External sources, tools, or future systems that are relevant to the autoresearch memory and orchestration roadmap.',
  'type': 'reference'}}

In [13]:
results_path = AUTORESEARCH_ROOT / 'results.tsv'
event_log_path = AUTORESEARCH_ROOT / 'experiment_memory.jsonl'
if not results_path.exists():
    raise RuntimeError('results.tsv is missing')
if not event_log_path.exists():
    raise RuntimeError('experiment_memory.jsonl is missing')

event_count = sum(1 for line in event_log_path.read_text(encoding='utf-8').splitlines() if line.strip())
print({
    'results_tsv': str(results_path),
    'experiment_memory_jsonl': str(event_log_path),
    'event_count': event_count,
})

{'results_tsv': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv', 'experiment_memory_jsonl': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/experiment_memory.jsonl', 'event_count': 2}


In [14]:
summary = {
    'memory_dir_exists': MEMORY_ROOT.exists(),
    'topic_file_count': len(expected_files) - 1,
    'has_exact_ledger': results_path.exists(),
    'has_append_only_event_log': event_log_path.exists(),
    'project_topic_types': sorted({item['type'] for item in metadata.values()}),
}
summary

{'memory_dir_exists': True,
 'topic_file_count': 4,
 'has_exact_ledger': True,
 'has_append_only_event_log': True,
 'project_topic_types': ['project', 'reference']}

## Success Criteria

This notebook is a successful smoke test when:

- the `memory/` directory exists
- `MEMORY.md` exists and links the expected topic files
- each topic file contains valid frontmatter with `name`, `description`, and `type`
- `results.tsv` still exists as the exact experiment ledger
- `experiment_memory.jsonl` still exists as the append-only event log

That confirms the project now has the intended hybrid short-term memory layout:

- exact operational records
- append-only event history
- small synthesized project memory
